# ADL Results Explorer

Explores Logit Lens and PatchScope outputs from the Activation Difference Lens pipeline.

In [3]:
from pathlib import Path
import matplotlib.pyplot as plt

# --- Configuration (edit these) ---
RESULTS_DIR = Path(
    "../../../adl_results/workspace/model-organisms/diffing_results/olmo2_1B/cake_bake_dpo_b0.05_lr1e-4_e1_r16/activation_difference_lens"
)
LAYERS = [7, 14, 15]
DATASET = "tulu-3-sft-olmo-2-mixture"
LOGIT_LENS_POSITION = -1  # Position for per-position logit lens view
PATCHSCOPE_POSITION = -1  # Position for per-position patchscope view
N_POSITIONS = 128  # Total positions (config: n)
LOGIT_LENS_MAX_ROWS = 20  # Set to an integer to truncate logit lens tables
PATCHSCOPE_GRADER = "openai_gpt-5-mini"
MODEL_ID = "allenai/OLMo-2-0425-1B-DPO"

LAYER_DIRS = {layer: RESULTS_DIR / f"layer_{layer}" / DATASET for layer in LAYERS}

In [4]:
import re
import torch
import pandas as pd
from collections import defaultdict
from transformers import AutoTokenizer

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 60)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)


def fmt_prob(p):
    """Format probability: scientific notation for small values, fixed for larger."""
    if abs(p) < 0.01:
        return f"{p:.2e}"
    return f"{p:.4f}"


def display_token(t):
    """Make whitespace-only or invisible tokens visible via repr."""
    if not t.strip():
        return repr(t)
    return t


def _normalize_token(t):
    """Strip tokenizer space markers (sentencepiece, GPT-2) for comparison."""
    return t.replace("\u2581", "").replace("\u0120", "").strip()


def load_logit_lens(layer, pos, prefix=""):
    """Load logit lens .pt file. Returns (top_k_probs, top_k_indices, inv_probs, inv_indices)."""
    return torch.load(
        LAYER_DIRS[layer] / f"{prefix}logit_lens_pos_{pos}.pt", weights_only=True
    )


def decode_tokens(indices):
    return [tokenizer.decode([int(i)]) for i in indices]


def load_patchscope(layer, pos, prefix=""):
    """Load auto_patch_scope .pt file. Returns dict with tokens_at_best_scale, selected_tokens, etc."""
    return torch.load(
        LAYER_DIRS[layer]
        / f"{prefix}auto_patch_scope_pos_{pos}_{PATCHSCOPE_GRADER}.pt",
        weights_only=False,
    )


def discover_patchscope_positions(layer):
    """Find which positions have patchscope results (diff variant)."""
    positions = []
    for f in sorted(
        LAYER_DIRS[layer].glob(f"auto_patch_scope_pos_*_{PATCHSCOPE_GRADER}.pt")
    ):
        m = re.search(r"auto_patch_scope_pos_(\d+)_", f.name)
        if m:
            positions.append(int(m.group(1)))
    return positions


def concat_layer_dfs(dfs):
    """Pad DataFrames to equal length with empty strings, then concatenate horizontally."""
    max_len = max(len(df) for df in dfs)
    padded = []
    for df in dfs:
        if len(df) < max_len:
            pad = pd.DataFrame(
                {col: [""] * (max_len - len(df)) for col in df.columns},
                index=range(len(df), max_len),
            )
            df = pd.concat([df, pad], axis=0)
        padded.append(df)
    return pd.concat(padded, axis=1)


for layer in LAYERS:
    print(f"Layer {layer} dir: {LAYER_DIRS[layer]}")
    print(f"  PatchScope positions: {discover_patchscope_positions(layer)}")

Layer 7 dir: ../../../adl_results/workspace/model-organisms/diffing_results/olmo2_1B/cake_bake_dpo_b0.05_lr1e-4_e1_r16/activation_difference_lens/layer_7/tulu-3-sft-olmo-2-mixture
  PatchScope positions: [0, 1, 2, 3, 4, 5]
Layer 14 dir: ../../../adl_results/workspace/model-organisms/diffing_results/olmo2_1B/cake_bake_dpo_b0.05_lr1e-4_e1_r16/activation_difference_lens/layer_14/tulu-3-sft-olmo-2-mixture
  PatchScope positions: [0, 1, 2, 3, 4, 5]
Layer 15 dir: ../../../adl_results/workspace/model-organisms/diffing_results/olmo2_1B/cake_bake_dpo_b0.05_lr1e-4_e1_r16/activation_difference_lens/layer_15/tulu-3-sft-olmo-2-mixture
  PatchScope positions: [0, 1, 2, 3, 4, 5]


## 1. Logit Lens Analysis

### 1A. Single Position

Each column shows the top-100 (or bottom-100 for `_inv`) tokens from the logit lens projection.  
Format: `token (softmax_prob)`

In [6]:
# Logit lens columns: (file prefix, tuple index for probs, tuple index for indices)
LL_VARIANTS = {
    "base": ("base_", 0, 1),
    "base_inv": ("base_", 2, 3),
    "ft": ("ft_", 0, 1),
    "ft_inv": ("ft_", 2, 3),
    "diff": ("", 0, 1),
    "diff_inv": ("", 2, 3),
}


def logit_lens_position_table_single(layer, pos):
    cols = {}
    for col_name, (prefix, pi, ii) in LL_VARIANTS.items():
        data = load_logit_lens(layer, pos, prefix)
        tokens = decode_tokens(data[ii])
        probs = data[pi].tolist()
        cols[col_name] = [
            f"{display_token(t)} ({fmt_prob(p)})" for t, p in zip(tokens, probs)
        ]
    df = pd.DataFrame(cols)
    if LOGIT_LENS_MAX_ROWS is not None:
        df = df.head(LOGIT_LENS_MAX_ROWS)
    return df


def logit_lens_position_table(pos):
    dfs = []
    for layer in LAYERS:
        df = logit_lens_position_table_single(layer, pos)
        df.columns = pd.MultiIndex.from_product([[f"layer_{layer}"], df.columns])
        dfs.append(df)
    return concat_layer_dfs(dfs)


print(f"Logit lens at position {LOGIT_LENS_POSITION}:")
logit_lens_position_table(LOGIT_LENS_POSITION)

Logit lens at position -1:


layer_7                                                \
                       base             base_inv                       ft   
0           .Today (0.0240)      urrenc (0.0216)          .Today (0.0256)   
1          .Second (0.0114)       pos (5.13e-03)         .Second (0.0121)   
2        Buccane (8.85e-03)       act (4.82e-03)       Buccane (9.46e-03)   
3          /Area (6.10e-03)    askell (3.52e-03)         /Area (6.47e-03)   
4            .au (4.88e-03)      gons (3.31e-03)           .au (4.73e-03)   
5            aru (3.94e-03)        �� (2.08e-03)           aru (4.46e-03)   
6      /problems (3.94e-03)      ejec (2.01e-03)     /problems (3.69e-03)   
7      /entities (2.96e-03)        دي (1.95e-03)     /entities (2.98e-03)   
8          /bind (2.70e-03)      azon (1.95e-03)         /Math (2.79e-03)   
9       /problem (2.62e-03)      anth (1.78e-03)         /bind (2.70e-03)   
10         /Math (2.62e-03)     fácil (1.78e-03)      /respond (2.62e-03)   
11      /respond (2.62e-03)     posix (1.72e-03)    confidence (2.62e-03)   
12          fter (2.46e-03)  essional (1.62e-03)      /problem (2.53e-03)   
13    confidence (2.30e-03)  Optional (1.56e-03)          ilot (2.17e-03)   
14   persistence (2.24e-03)    Phones (1.47e-03)   persistence (2.17e-03)   
15     /activity (2.24e-03)      Vers (1.47e-03)          fter (2.17e-03)   
16     /operator (2.24e-03)        vs (1.38e-03)     .AddRange (2.04e-03)   
17          ilot (1.98e-03)       med (1.26e-03)     /activity (2.04e-03)   
18     belonging (1.98e-03)  >Welcome (1.26e-03)          oire (1.91e-03)   
19          oire (1.85e-03)      orst (1.26e-03)     /operator (1.85e-03)   

                                                                      \
                 ft_inv                    diff             diff_inv   
0       urrenc (0.0254)          rix (8.00e-03)        Mort (0.0114)   
1        pos (5.49e-03)          bst (8.00e-03)       avi (8.85e-03)   
2        act (4.70e-03)      Expires (6.23e-03)      ames (4.46e-03)   
3       gons (3.66e-03)       répond (4.70e-03)       MPI (4.46e-03)   
4     askell (3.56e-03)         itar (4.70e-03)     ágina (4.46e-03)   
5         �� (2.29e-03)          ney (4.27e-03)      PLIT (4.18e-03)   
6       ejec (2.15e-03)         ansa (4.03e-03)       IDS (3.69e-03)   
7         دي (2.03e-03)          Ein (3.34e-03)       efe (3.46e-03)   
8      fácil (1.90e-03)      Dropout (3.23e-03)      NING (3.05e-03)   
9   Optional (1.79e-03)          bra (3.04e-03)    Picker (3.05e-03)   
10     posix (1.79e-03)         trag (2.94e-03)       abe (2.61e-03)   
11  essional (1.73e-03)        retty (2.94e-03)   dernier (2.46e-03)   
12      azon (1.68e-03)   strategist (2.94e-03)       Hut (2.38e-03)   
13      Vers (1.48e-03)         .art (2.85e-03)   Tiffany (2.30e-03)   
14      anth (1.48e-03)          ill (2.52e-03)      midd (2.24e-03)   
15    Phones (1.43e-03)         atos (2.23e-03)         � (2.17e-03)   
16  >Welcome (1.35e-03)         ento (2.15e-03)   agascar (2.17e-03)   
17      orst (1.30e-03)         Pont (2.15e-03)       afr (2.17e-03)   
18        vs (1.27e-03)      Grocery (2.15e-03)     -mort (2.04e-03)   
19       med (1.19e-03)      answers (2.15e-03)    sentir (2.04e-03)   

                layer_14                                               \
                    base               base_inv                    ft   
0            To (0.9062)          zoek (0.8555)           To (0.9102)   
1           The (0.0479)      contador (0.1309)          The (0.0376)   
2           Let (0.0156)           메 (8.36e-03)          Let (0.0177)   
3            In (0.0146)         иск (3.49e-03)           In (0.0138)   
4         ### (4.46e-03)     Produto (2.12e-03)        ### (6.53e-03)   
5           A (2.88e-03)           � (1.42e-05)          A (3.97e-03)   
6         For (1.27e-03)      Resets (9.83e-06)        For (1.41e-03)   
7          As (9.92e-04)     Detalle (9.83e-06)          I (9.69e-04)   
8         

In [7]:
# Logit lens columns: (file prefix, tuple index for probs, tuple index for indices)
LL_VARIANTS = {
    "base": ("base_", 0, 1),
    "base_inv": ("base_", 2, 3),
    "ft": ("ft_", 0, 1),
    "ft_inv": ("ft_", 2, 3),
    "diff": ("", 0, 1),
    "diff_inv": ("", 2, 3),
}


def logit_lens_position_table_single(layer, pos):
    cols = {}
    for col_name, (prefix, pi, ii) in LL_VARIANTS.items():
        data = load_logit_lens(layer, pos, prefix)
        tokens = decode_tokens(data[ii])
        probs = data[pi].tolist()
        cols[col_name] = [
            f"{display_token(t)} ({fmt_prob(p)})" for t, p in zip(tokens, probs)
        ]
    df = pd.DataFrame(cols)
    if LOGIT_LENS_MAX_ROWS is not None:
        df = df.head(LOGIT_LENS_MAX_ROWS)
    return df


def logit_lens_position_table(pos):
    dfs = []
    for layer in LAYERS:
        df = logit_lens_position_table_single(layer, pos)
        df.columns = pd.MultiIndex.from_product([[f"layer_{layer}"], df.columns])
        dfs.append(df)
    return concat_layer_dfs(dfs)


print(f"Logit lens at position {5}:")
logit_lens_position_table(5)

Logit lens at position 5:


layer_7                                                \
                       base             base_inv                       ft   
0         /problem (0.0398)       .vn (7.81e-03)        /problem (0.0381)   
1        /entities (0.0265)       (us (5.04e-03)       /entities (0.0270)   
2        /problems (0.0171)      sagt (4.46e-03)       /problems (0.0179)   
3         .Today (9.16e-03)       ]]; (3.94e-03)        .Today (9.03e-03)   
4        /global (8.61e-03)        że (3.46e-03)       /global (8.79e-03)   
5        /manage (7.87e-03)    testim (2.87e-03)       /manage (8.79e-03)   
6           /job (6.71e-03)       ')" (2.70e-03)          /job (7.72e-03)   
7   /preferences (6.10e-03)       ($. (2.70e-03)       /layout (6.23e-03)   
8        /layout (5.74e-03)      -ves (2.70e-03)  /preferences (6.01e-03)   
9      /provider (5.07e-03)     zeigt (2.53e-03)       /crypto (4.55e-03)   
10       /crypto (4.61e-03)     spons (2.24e-03)   /connection (4.55e-03)   
11   /connection (4.21e-03)     feliz (2.24e-03)          /reg (4.00e-03)   
12    WHATSOEVER (4.06e-03)     lesen (2.11e-03)       /engine (4.00e-03)   
13      /logging (3.94e-03)   kontrol (1.97e-03)     /provider (4.00e-03)   
14  /environment (3.94e-03)       (!! (1.97e-03)  /environment (4.00e-03)   
15       /engine (3.83e-03)    spiele (1.97e-03)    WHATSOEVER (3.89e-03)   
16          /reg (3.71e-03)      helf (1.74e-03)        .Round (3.54e-03)   
17       /entity (3.48e-03)     scrut (1.54e-03)      /logging (3.54e-03)   
18      /effects (3.37e-03)       )": (1.44e-03)      /weather (3.22e-03)   
19       /dialog (3.37e-03)    mostra (1.44e-03)      /effects (3.22e-03)   

                                                                           \
                 ft_inv                       diff               diff_inv   
0        .vn (7.87e-03)              Pont (0.0139)          Mort (0.1289)   
1        (us (5.77e-03)            valign (0.0101)         avi (8.24e-03)   
2       sagt (4.21e-03)              ento (0.0101)        crem (5.49e-03)   
3         że (3.72e-03)            quiv (8.97e-03)     agascar (5.16e-03)   
4        ]]; (3.49e-03)        levation (7.93e-03)       hairs (4.27e-03)   
5       -ves (2.90e-03)             bst (6.99e-03)         Moh (4.15e-03)   
6     testim (2.90e-03)            itar (6.56e-03)        NING (3.91e-03)   
7        ($. (2.72e-03)              yt (4.24e-03)     dernier (3.66e-03)   
8        ')" (2.72e-03)            obed (3.51e-03)        jing (3.34e-03)   
9      feliz (2.56e-03)           Blick (3.10e-03)   trembling (3.04e-03)   
10     zeigt (2.56e-03)            ered (3.10e-03)    Facility (2.85e-03)   
11     spons (2.40e-03)             ney (2.73e-03)      ISTORY (2.52e-03)   
12     lesen (2.26e-03)   Internacional (2.06e-03)        ames (2.37e-03)   
13   kontrol (2.12e-03)     unsubscribe (2.06e-03)        mort (1.95e-03)   
14       (!! (2.00e-03)            teil (1.82e-03)        PLIT (1.95e-03)   
15    spiele (1.75e-03)           ículo (1.82e-03)      Butter (1.95e-03)   
16     scrut (1.65e-03)           uchar (1.82e-03)       usado (1.90e-03)   
17      helf (1.65e-03)         Catalog (1.76e-03)           ذ (1.84e-03)   
18       fas (1.55e-03)         proport (1.71e-03)         isl (1.84e-03)   
19    mostra (1.46e-03)          ottage (1.66e-03)         jar (1.73e-03)   

            layer_14                                          \
                base              base_inv                ft   
0         , (0.5898)     contador (0.8516)        , (0.5430)   
1       and (0.1465)    kontrol (7.39e-03)      and (0.1641)   
2       the (0.1270)         �� (5.74e-03)      the (0.1611)   
3        in (0.0569)   karakter (5.74e-03)      ' ' (0.0508)   
4       ' ' (0.0479)         bö (5.74e-03)       in (0.0491)   
5         a (0.0129)         �� (5.74e-03)        a (0.0150)   
6       ( (3.31e-03)      KANJI (3.48e-03)     of (2.59e-03)   
7      to (2.99e-03)      subur (3.48e-03)      ( (2.5

### 1B. Aggregated Across All Positions

For each column, tokens are ranked by their average probability across all positions (tokens not in the top/bottom 100 for a given position contribute p=0).  
Format: `token (avg_prob)`

In [8]:
def logit_lens_aggregated_single(layer):
    agg = {}
    for col_name, (prefix, pi, ii) in LL_VARIANTS.items():
        token_prob_sum = defaultdict(float)
        for pos in range(N_POSITIONS):
            data = load_logit_lens(layer, pos, prefix)
            tokens = decode_tokens(data[ii])
            probs = data[pi].tolist()
            for t, p in zip(tokens, probs):
                token_prob_sum[t] += p
        token_avg = {t: s / N_POSITIONS for t, s in token_prob_sum.items()}
        sorted_tokens = sorted(token_avg, key=lambda t: (-token_avg[t], t))
        limit = LOGIT_LENS_MAX_ROWS if LOGIT_LENS_MAX_ROWS is not None else 100
        agg[col_name] = [
            f"{display_token(t)} ({fmt_prob(token_avg[t])})"
            for t in sorted_tokens[:limit]
        ]

    max_len = max(len(v) for v in agg.values())
    for k in agg:
        agg[k] += [""] * (max_len - len(agg[k]))
    return pd.DataFrame(agg)


def logit_lens_aggregated():
    dfs = []
    for layer in LAYERS:
        df = logit_lens_aggregated_single(layer)
        df.columns = pd.MultiIndex.from_product([[f"layer_{layer}"], df.columns])
        dfs.append(df)
    return concat_layer_dfs(dfs)


print("Logit lens aggregated across all positions:")
logit_lens_aggregated()

Logit lens aggregated across all positions:


layer_7                                                \
                       base             base_inv                       ft   
0        /entities (0.0257)         .vn (0.0195)       /entities (0.0259)   
1         /problem (0.0139)   /Register (0.0114)        /problem (0.0133)   
2      /problems (9.19e-03)    testim (6.97e-03)     /problems (9.41e-03)   
3        /global (6.68e-03)      sagt (6.57e-03)       /global (6.91e-03)   
4   /environment (5.95e-03)     asign (5.31e-03)       /manage (6.48e-03)   
5      /provider (5.86e-03)       -ie (4.91e-03)   /connection (6.20e-03)   
6         .Today (5.79e-03)     zeigt (4.03e-03)  /environment (6.04e-03)   
7    /connection (5.73e-03)        że (3.98e-03)        .Today (5.77e-03)   
8        /manage (5.63e-03)      -ves (3.30e-03)     /customer (4.68e-03)   
9      /customer (4.73e-03)         ť (2.95e-03)     /provider (4.61e-03)   
10  /preferences (4.25e-03)   personn (2.83e-03)  /preferences (4.14e-03)   
11       /dialog (3.36e-03)     probs (2.78e-03)       /shared (3.97e-03)   
12       /shared (3.34e-03)      elig (2.59e-03)      libertin (3.34e-03)   
13      /account (3.22e-03)      roku (2.35e-03)       /dialog (3.22e-03)   
14       /entity (3.18e-03)    ):\n\n (2.35e-03)      /account (3.13e-03)   
15      libertin (3.10e-03)     lesen (2.30e-03)       /layout (3.10e-03)   
16       /layout (2.91e-03)  ,,,,,,,, (2.24e-03)       /entity (2.98e-03)   
17          .Try (2.83e-03)       )": (2.20e-03)          .Try (2.84e-03)   
18      /effects (2.71e-03)    spiele (2.11e-03)        /legal (2.53e-03)   
19        /legal (2.65e-03)       (us (2.09e-03)       /crypto (2.49e-03)   

                                                     \
                  ft_inv                       diff   
0           .vn (0.0200)               bst (0.0448)   
1   /Register (8.80e-03)          levation (0.0125)   
2      testim (7.00e-03)              Pont (0.0116)   
3        sagt (6.45e-03)          valign (7.76e-03)   
4         -ie (4.65e-03)            quiv (6.08e-03)   
5       asign (4.64e-03)            ento (5.68e-03)   
6          że (4.58e-03)            itar (5.59e-03)   
7       zeigt (4.04e-03)             spo (2.83e-03)   
8        -ves (3.66e-03)           baugh (2.79e-03)   
9           ť (2.81e-03)              ía (2.79e-03)   
10    personn (2.72e-03)            teil (2.53e-03)   
11      probs (2.69e-03)            bout (2.48e-03)   
12       elig (2.56e-03)         ---\n\n (2.45e-03)   
13   ,,,,,,,, (2.49e-03)        abilidad (2.38e-03)   
14      lesen (2.45e-03)           Blick (2.37e-03)   
15        esl (2.40e-03)            erie (2.34e-03)   
16     ):\n\n (2.39e-03)            edit (2.26e-03)   
17        (us (2.34e-03)         .google (2.13e-03)   
18       roku (2.26e-03)   Internacional (2.03e-03)   
19      thous (2.17e-03)          sector (1.98e-03)   

                                          layer_14                        \
                       diff_inv               base              base_inv   
0                 Mort (0.0784)         , (0.8019)     contador (0.9621)   
1                 ız (5.40e-03)       ' ' (0.1086)    kontrol (3.15e-03)   
2               ames (4.67e-03)       the (0.0409)   karakter (2.50e-03)   
3             ISTORY (4.44e-03)       and (0.0305)       rekl (1.59e-03)   
4              ATORY (3.97e-03)      in (6.08e-03)         bö (1.38e-03)   
5          condition (3.88e-03)       ( (4.50e-03)       zoek (1.16e-03)   
6               jing (3.18e-03)      's (2.96e-03)     testim (9.65e-04)   
7                avi (3.15e-03)       a (1.69e-03)    Produto (9.60e-04)   
8               crem (2.87e-03)      to (9.80e-04)     bilder (8.68e-04)   
9              Ihrem (2.54e-03)       . (6.77e-04)         �� (7.72e-04)   
10  ________________ (2.46e-03)      of (4.03e-04)       dara (7.54e-04)   
11               isl (2.43e-03)     for (1.77e-04)       helf (6.87e-04)   
12             hairs (2.39e-03)      is (6.84e-05)

## 2. PatchScope Analysis

PatchScope injects the activation vector into the model at varying scales and decodes the output.  
Unlike logit lens, there are no inverse variants -- only `base`, `ft`, and `diff`.  
Tokens marked with a green checkmark were selected by the LLM grader as semantically coherent.

### 2A. Single Position

Shows tokens at the best scale found by the auto patch scope search.  
Format: `token (prob)` with `\u2705` if in `selected_tokens`

In [9]:
PS_VARIANTS = [("base", "base_"), ("ft", "ft_"), ("diff", "")]


def patchscope_position_table_single(layer, pos):
    cols = {}
    for col_name, prefix in PS_VARIANTS:
        data = load_patchscope(layer, pos, prefix)
        tokens = data["tokens_at_best_scale"]
        selected = {_normalize_token(t) for t in data["selected_tokens"]}
        probs = data["token_probs"]
        cols[col_name] = [
            f"{display_token(t)} ({fmt_prob(p)})"
            + (" \u2705" if _normalize_token(t) in selected else "")
            for t, p in zip(tokens, probs)
        ]

    max_len = max(len(v) for v in cols.values())
    for k in cols:
        cols[k] += [""] * (max_len - len(cols[k]))
    return pd.DataFrame(cols)


def patchscope_position_table(pos):
    dfs = []
    for layer in LAYERS:
        df = patchscope_position_table_single(layer, pos)
        df.columns = pd.MultiIndex.from_product([[f"layer_{layer}"], df.columns])
        dfs.append(df)
    return concat_layer_dfs(dfs)


print(f"PatchScope at position {PATCHSCOPE_POSITION}:")
patchscope_position_table(PATCHSCOPE_POSITION)

PatchScope at position -1:


layer_7                           \
                       base                       ft   
0           .Today (0.0258)          .Today (0.0263)   
1        .Second (9.68e-03)       .Second (0.0102) ✅   
2        Buccane (6.52e-03)       Buccane (6.99e-03)   
3        /Area (5.70e-03) ✅       /Area (6.05e-03) ✅   
4            .au (5.29e-03)           aru (5.69e-03)   
5            aru (5.03e-03)           .au (5.17e-03)   
6    /problems (3.17e-03) ✅    confidence (3.14e-03)   
7     confidence (2.86e-03)   /problems (3.01e-03) ✅   
8           fter (2.77e-03)          ilot (2.77e-03)   
9    /entities (2.45e-03) ✅       /Math (2.60e-03) ✅   
10       /Math (2.45e-03) ✅          fter (2.45e-03)   
11          ilot (2.45e-03)   /entities (2.45e-03) ✅   
12       /bind (2.16e-03) ✅       /bind (2.16e-03) ✅   
13    /problem (2.16e-03) ✅    /problem (2.03e-03) ✅   
14   /activity (2.10e-03) ✅   /activity (1.96e-03) ✅   
15    /respond (1.96e-03) ✅    /respond (1.96e-03) ✅   
16   persistence (1.96e-03)   persistence (1.90e-03)   
17   /operator (1.90e-03) ✅   .AddRange (1.72e-03) ✅   
18     belonging (1.89e-03)     belonging (1.68e-03)   
19     .AddRange (1.58e-03)   /operator (1.58e-03) ✅   

                                            layer_14                          \
                        diff                    base                      ft   
0              itar (0.0140)             To (0.9141)             To (0.9141)   
1               rix (0.0109)            The (0.0427)            The (0.0334)   
2             bst (9.44e-03)          Let (0.0167) ✅            Let (0.0190)   
3             ney (8.08e-03)             In (0.0115)             In (0.0115)   
4            ansa (6.91e-03)          ### (5.77e-03)          ### (8.61e-03)   
5         Dropout (5.85e-03)            A (2.04e-03)            A (2.73e-03)   
6           retty (4.32e-03)           ** (1.07e-03)           ** (1.21e-03)   
7         Expires (4.02e-03)          For (9.64e-04)          For (1.07e-03)   
8             bra (3.70e-03)           As (7.82e-04)  Certainly (9.16e-04) ✅   
9            trag (3.47e-03)            I (6.90e-04)            I (7.36e-04)   
10            Ein (3.36e-03)            1 (6.48e-04)           As (7.36e-04)   
11       répond (3.19e-03) ✅  Certainly (5.72e-04) ✅            1 (5.91e-04)   
12            ill (3.03e-03)       Sure (4.55e-04) ✅       Sure (5.38e-04) ✅   
13           mlin (2.85e-03)           We (3.93e-04)           We (3.81e-04)   
14           Pont (2.82e-03)      First (3.70e-04) ✅        First (3.81e-04)   
15           heat (2.74e-03)           It (3.26e-04)           It (2.98e-04)   
16              � (2.71e-03)       This (3.06e-04) ✅        Given (2.71e-04)   
17          aways (2.66e-03)      Given (2.02e-04) ✅         This (2.17e-04)   
18   strategist (2.66e-03) ✅            S (1.28e-04)         Here (1.50e-04)   
19      answers (2.22e-03) ✅       Here (1.28e-04) ✅            H (1.20e-04)   

                                        layer_15                        \
                      diff                  base                    ft   
0      contador (0.5690) ✅           To (0.4141)           ** (0.3984)   
1               메 (0.1348)           ** (0.2852)          ### (0.3105)   
2       Produto (0.0165) ✅          ### (0.2500)           To (0.2422)   
3        zoek (5.44e-03) ✅        Let (0.0265) ✅        Let (0.0327) ✅   
4       lokal (4.99e-03) ✅          The (0.0160)        The (6.44e-03)   
5          spos (4.17e-03)  Certainly (2.46e-03)  Certainly (3.91e-03)   
6          inne (2.64e-03)       Sure (1.49e-03)       Sure (1.63e-03)   
7           ありが (2.43e-03)         In (1.16e-03)         ## (9.92e-04)   
8         cpf (2.27e-03) ✅         ## (7.97e-04)         In (7.71e-04)   
9     kontrol (2.25e-03) ✅    Given (3.78e-04) ✅    Given (4.12e-04) ✅   
10          poj (2.24e-03)        1 (2.29e-04) ✅        ``` (2.82e-04)   
11       kode (2.19e-03) ✅    First (2.29e-04) ✅       #### (2.2

### 2B. Aggregated Across All PatchScope Positions

Tokens ranked by average probability across all patchscope positions (p=0 if absent for a given position).  
Green checkmark if the token was in `selected_tokens` for **any** position.  
Format: `token (avg_prob)`

In [10]:
def patchscope_aggregated_single(layer):
    ps_positions = discover_patchscope_positions(layer)
    n_ps = len(ps_positions)

    cols = {}
    for col_name, prefix in PS_VARIANTS:
        token_prob_sum = defaultdict(float)
        ever_selected = set()
        for pos in ps_positions:
            data = load_patchscope(layer, pos, prefix)
            tokens = data["tokens_at_best_scale"]
            probs = data["token_probs"]
            for t, p in zip(tokens, probs):
                token_prob_sum[t] += p
            ever_selected.update(_normalize_token(t) for t in data["selected_tokens"])

        token_avg = {t: s / n_ps for t, s in token_prob_sum.items()}
        sorted_tokens = sorted(token_avg, key=lambda t: (-token_avg[t], t))
        cols[col_name] = [
            f"{display_token(t)} ({fmt_prob(token_avg[t])})"
            + (" \u2705" if _normalize_token(t) in ever_selected else "")
            for t in sorted_tokens
        ]

    max_len = max(len(v) for v in cols.values())
    for k in cols:
        cols[k] += [""] * (max_len - len(cols[k]))
    return pd.DataFrame(cols)


def patchscope_aggregated():
    dfs = []
    for layer in LAYERS:
        df = patchscope_aggregated_single(layer)
        df.columns = pd.MultiIndex.from_product([[f"layer_{layer}"], df.columns])
        dfs.append(df)
    return concat_layer_dfs(dfs)


ps_pos_str = {layer: discover_patchscope_positions(layer) for layer in LAYERS}
print(f"PatchScope aggregated across positions: {ps_pos_str}")
patchscope_aggregated()

PatchScope aggregated across positions: {7: [0, 1, 2, 3, 4, 5], 14: [0, 1, 2, 3, 4, 5], 15: [0, 1, 2, 3, 4, 5]}


layer_7                              \
                         base                          ft   
0                 -> (0.0527)          problem (0.0444) ✅   
1                 's (0.0316)              :\n\n (0.0349)   
2         /problem (0.0213) ✅                 's (0.0326)   
3              :\n\n (0.0209)                the (0.0268)   
4            solve (0.0200) ✅            solve (0.0252) ✅   
5             '\n\n' (0.0187)         /problem (0.0243) ✅   
6        /problems (0.0159) ✅        /entities (0.0162) ✅   
7        /entities (0.0154) ✅        /problems (0.0139) ✅   
8          /manage (0.0124) ✅              seems (0.0103)   
9                the (0.0117)        /manage (9.05e-03) ✅   
10             you (8.65e-03)              :\n (8.07e-03)   
11            '\n' (8.54e-03)             this (8.02e-03)   
12               , (8.43e-03)         solves (6.08e-03) ✅   
13           seems (5.97e-03)              you (4.95e-03)   
14    understand (5.87e-03) ✅               -> (4.84e-03)   
15             :\n (5.74e-03)             your (4.24e-03)   
16     statement (4.40e-03) ✅         .Today (4.03e-03) ✅   
17        .Today (3.79e-03) ✅     understand (3.95e-03) ✅   
18        solves (3.39e-03) ✅           math (3.59e-03) ✅   
19       analyze (3.35e-03) ✅       problems (3.32e-03) ✅   
20      involves (3.13e-03) ✅   /preferences (3.29e-03) ✅   
21       address (3.11e-03) ✅           '\n\n' (3.19e-03)   
22  /preferences (3.06e-03) ✅       question (3.16e-03) ✅   
23              ’s (2.85e-03)        address (3.11e-03) ✅   
24              is (2.78e-03)           /job (3.05e-03) ✅   
25      requires (2.77e-03) ✅        /global (3.00e-03) ✅   
26       /global (2.76e-03) ✅               ’s (2.65e-03)   
27       /crypto (2.63e-03) ✅        analyze (2.61e-03) ✅   
28            your (2.44e-03)                , (2.57e-03)   
29   /connection (2.35e-03) ✅         tackle (2.32e-03) ✅   
30     /provider (2.34e-03) ✅        /crypto (2.26e-03) ✅   
31        tackle (2.12e-03) ✅        /layout (1.89e-03) ✅   
32         break (1.86e-03) ✅        /engine (1.84e-03) ✅   
33               : (1.70e-03)        /object (1.81e-03) ✅   
34       /shared (1.60e-03) ✅         puzzle (1.68e-03) ✅   
35          /job (1.50e-03) ✅    /connection (1.64e-03) ✅   
36       /object (1.42e-03) ✅   /environment (1.59e-03) ✅   
37              we (1.41e-03)      /provider (1.49e-03) ✅   
38       /layout (1.35e-03) ✅          appears (1.37e-03)   
39       /engine (1.34e-03) ✅             '\n' (1.23e-03)   
40      /logging (1.34e-03) ✅            looks (1.17e-03)   
41             /pr (1.27e-03)        /shared (1.12e-03) ✅   
42        decode (1.17e-03) ✅                : (1.08e-03)   
43          begins (1.14e-03)       /logging (1.08e-03) ✅   
44  /controllers (1.03e-03) ✅       /effects (1.03e-03) ✅   
45          .Round (9.93e-04)          break (1.01e-03) ✅   
46      /effects (9.87e-04) ✅         decode (9.88e-04) ✅   
47     /activity (9.60e-04) ✅         solved (9.71e-04) ✅   
48           /pl (9.13e-04) ✅   /application (9.66e-04) ✅   
49      solution (8.97e-04) ✅         begins (9.55e-04) ✅   
50          step (8.43e-04) ✅      /activity (9.53e-04) ✅   
51             ' ' (8.18e-04)         /legal (9.46e-04) ✅   
52           looks (8.17e-04)        /entity (9.27e-04) ✅   
53            /con (8.10e-04)      statement (9.21e-04) ✅   
54         appears (7.81e-04)            these (9.21e-04)   
55           .\n\n (7.68e-04)   mathematical (9.10e-04) ✅   
56              in (7.54e-04)         coding (7.39e-04) ✅   
57           :\n\n (7.33e-04)        puzzles (7.38e-04) ✅   
58  /environment (7.17e-04) ✅    programming (7.36e-04) ✅   
59      question (7.15e-04) ✅              /pr (6.92e-04)   
60      /testing (6.51e-04) ✅       /testing (6.51e-04) ✅   
61          /reg (5.89e-04) ✅           /reg (6.36e-04) ✅   
62      /respond (5.40e-04) ✅           .Round (5.39e-04)   
63        /tasks (5.18e-04) ✅         /tasks (5.19e-04) ✅

## 3. Diff Logit Lens Across Positions

Shows only the **diff** variant of the logit lens for selected positions across all layers.
Format: `token (softmax_prob)`

In [11]:
DIFF_POSITIONS = [-3, -1, 0, 1, 2, 3, 10, 50, 100]


def logit_lens_diff_positions_table():
    """Show diff logit lens across multiple positions for all layers."""
    dfs = []
    for layer in LAYERS:
        col_data = {}
        for pos in DIFF_POSITIONS:
            prefix, pi, ii = LL_VARIANTS["diff"]
            data = load_logit_lens(layer, pos, prefix)
            tokens = decode_tokens(data[ii])
            probs = data[pi].tolist()
            col = [f"{display_token(t)} ({fmt_prob(p)})" for t, p in zip(tokens, probs)]
            if LOGIT_LENS_MAX_ROWS is not None:
                col = col[:LOGIT_LENS_MAX_ROWS]
            col_data[f"pos_{pos}"] = col
        layer_df = pd.DataFrame(col_data)
        layer_df.columns = pd.MultiIndex.from_product(
            [[f"layer_{layer}"], layer_df.columns]
        )
        dfs.append(layer_df)
    return concat_layer_dfs(dfs)


print(f"Logit lens DIFF across positions: {DIFF_POSITIONS}")
logit_lens_diff_positions_table()

Logit lens DIFF across positions: [-3, -1, 0, 1, 2, 3, 10, 50, 100]


layer_7                                                     \
                 pos_-3                  pos_-1                      pos_0   
0          rix (0.0141)          rix (8.00e-03)              Pont (0.0110)   
1          bnb (0.0103)          bst (8.00e-03)             bst (5.71e-03)   
2   altitude (7.57e-03)      Expires (6.23e-03)              yt (5.37e-03)   
3         уг (5.52e-03)       répond (4.70e-03)             ent (5.22e-03)   
4    datable (5.19e-03)         itar (4.70e-03)            bred (5.04e-03)   
5      itere (4.88e-03)          ney (4.27e-03)             ель (4.06e-03)   
6        ост (4.88e-03)         ansa (4.03e-03)            .art (3.94e-03)   
7       idad (4.88e-03)          Ein (3.34e-03)           artic (3.94e-03)   
8        *)" (4.30e-03)      Dropout (3.23e-03)            reta (3.46e-03)   
9      paren (3.80e-03)          bra (3.04e-03)   Internacional (3.07e-03)   
10     erner (3.36e-03)         trag (2.94e-03)              yc (2.79e-03)   
11       AIM (2.78e-03)        retty (2.94e-03)             ney (2.70e-03)   
12   ?family (2.78e-03)   strategist (2.94e-03)            vent (2.70e-03)   
13    erness (2.61e-03)         .art (2.85e-03)            ento (2.53e-03)   
14     abric (2.46e-03)          ill (2.52e-03)            Iron (2.46e-03)   
15   Aligned (2.46e-03)         atos (2.23e-03)              oy (2.38e-03)   
16         신 (2.30e-03)         ento (2.15e-03)            aran (2.30e-03)   
17      itel (2.30e-03)         Pont (2.15e-03)             bra (2.30e-03)   
18      /)\n (2.17e-03)      Grocery (2.15e-03)             spo (2.24e-03)   
19   usahaan (2.17e-03)      answers (2.15e-03)            ered (2.24e-03)   

                                                                         \
                        pos_1                pos_2                pos_3   
0               Pont (0.0121)      Pont (9.83e-03)      quiv (8.54e-03)   
1               yt (8.30e-03)      ered (7.63e-03)      Pont (8.06e-03)   
2             ered (5.71e-03)      quiv (7.17e-03)      ento (7.57e-03)   
3              bst (4.73e-03)       bst (6.74e-03)  levation (7.57e-03)   
4             ento (4.73e-03)      ento (5.58e-03)       bst (6.68e-03)   
5    Internacional (3.94e-03)      itar (5.25e-03)    valign (5.52e-03)   
6            artic (3.94e-03)    valign (5.25e-03)      itar (4.30e-03)   
7               yc (3.69e-03)  levation (4.09e-03)     uchar (4.30e-03)   
8             itar (3.36e-03)    erness (3.62e-03)       ney (3.57e-03)   
9         levation (3.16e-03)        yt (3.39e-03)      ered (3.57e-03)   
10          valign (2.70e-03)     uchar (2.99e-03)     (...) (3.57e-03)   
11            quiv (2.62e-03)      obed (2.73e-03)      obed (2.61e-03)   
12           ences (2.46e-03)       ney (2.49e-03)     Blick (2.61e-03)   
13          resent (2.46e-03)     (...) (2.33e-03)        yt (2.53e-03)   
14           _cast (2.30e-03)       bra (2.33e-03)       yat (1.85e-03)   
15            bred (2.04e-03)      reta (2.26e-03)      teil (1.74e-03)   
16            reta (1.98e-03)       yat (2.12e-03)     itori (1.69e-03)   
17             ney (1.98e-03)      iros (2.12e-03)      iros (1.59e-03)   
18             yat (1.85e-03)       ель (1.82e-03)      heat (1.59e-03)   
19          erness (1.69e-03)   proport (1.76e-03)   proport (1.49e-03)   

                                                                         \
                       pos_10               pos_50              pos_100   
0               Pont (0.0160)         bst (0.0427)         bst (0.0601)   
1             valign (0.0110)    levation (0.0148)    levation (0.0126)   
2                bst (0.0103)        Pont (0.0108)      Pont (9.22e-03)   
3             quiv (8.06e-03)    valign (8.97e-03)    valign (5.95e-03)   
4         levation (7.08e-03)      itar (6.56e-03)      itar (5.25e-03)   
5             ento (6.65e-03)      ento (6.16e-03)      quiv (4.64e-03)   
6             itar (4.30e-03)      quiv (6.16e-0

## 4. Diff PatchScope Across Positions

Shows only the **diff** variant of PatchScope for selected positions across all layers.
Format: `token (prob)` with `✅` if in `selected_tokens`

In [12]:
PS_DIFF_POSITIONS = [-3, -1, 0, 1, 2, 3]


def patchscope_diff_positions_table():
    """Show diff patchscope across multiple positions for all layers."""
    dfs = []
    for layer in LAYERS:
        col_data = {}
        for pos in PS_DIFF_POSITIONS:
            try:
                data = load_patchscope(layer, pos, prefix="")
            except FileNotFoundError:
                col_data[f"pos_{pos}"] = ["(not available)"]
                continue
            tokens = data["tokens_at_best_scale"]
            selected = {_normalize_token(t) for t in data["selected_tokens"]}
            probs = data["token_probs"]
            col = [
                f"{display_token(t)} ({fmt_prob(p)})"
                + (" ✅" if _normalize_token(t) in selected else "")
                for t, p in zip(tokens, probs)
            ]
            col_data[f"pos_{pos}"] = col
        layer_df = pd.DataFrame({k: pd.Series(v) for k, v in col_data.items()}).fillna(
            ""
        )
        layer_df.columns = pd.MultiIndex.from_product(
            [[f"layer_{layer}"], layer_df.columns]
        )
        dfs.append(layer_df)
    return concat_layer_dfs(dfs)


print(f"PatchScope DIFF across positions: {DIFF_POSITIONS}")
patchscope_diff_positions_table()

PatchScope DIFF across positions: [-3, -1, 0, 1, 2, 3, 10, 50, 100]


layer_7                                                       \
                 pos_-3                    pos_-1                      pos_0   
0           ol (0.0200)             itar (0.0140)              Pont (0.0105)   
1           ap (0.0172)              rix (0.0109)        Plymouth (9.75e-03)   
2       asso (0.0114) ✅            bst (9.44e-03)            itar (8.87e-03)   
3          aan (0.0112)            ney (8.08e-03)             ega (4.70e-03)   
4         JI (8.68e-03)           ansa (6.91e-03)           uelle (4.56e-03)   
5        van (8.50e-03)        Dropout (5.85e-03)             ney (4.45e-03)   
6     assa (6.98e-03) ✅          retty (4.32e-03)            Vale (4.15e-03)   
7        era (6.51e-03)        Expires (4.02e-03)            reta (3.86e-03)   
8         ra (6.46e-03)            bra (3.70e-03)            ento (3.80e-03)   
9        hud (6.41e-03)           trag (3.47e-03)   PlayStation (3.76e-03) ✅   
10    conf (5.32e-03) ✅            Ein (3.36e-03)             spo (3.49e-03)   
11         q (5.13e-03)       répond (3.19e-03) ✅            bred (3.39e-03)   
12       'Re (4.63e-03)            ill (3.03e-03)        sector (3.23e-03) ✅   
13         p (4.07e-03)           mlin (2.85e-03)           cam (3.12e-03) ✅   
14        ts (3.72e-03)           Pont (2.82e-03)            vent (2.97e-03)   
15     arser (3.40e-03)           heat (2.74e-03)        bsites (2.54e-03) ✅   
16     ass (3.35e-03) ✅              � (2.71e-03)            yt (2.52e-03) ✅   
17       ira (3.31e-03)          aways (2.66e-03)         Level (2.48e-03) ✅   
18         v (3.16e-03)   strategist (2.66e-03) ✅        Santiago (2.45e-03)   
19  anning (2.89e-03) ✅      answers (2.22e-03) ✅            iros (2.41e-03)   

                                                                     \
                     pos_1            pos_2                   pos_3   
0    Plymouth (9.16e-03) ✅    28 (0.0503) ✅           itar (0.0219)   
1          iros (8.56e-03)    48 (0.0181) ✅           quiv (0.0106)   
2        bsites (8.48e-03)    42 (0.0132) ✅         iros (9.64e-03)   
3             ů (6.44e-03)    38 (0.0129) ✅     levation (6.88e-03)   
4          quiv (6.16e-03)  ento (8.72e-03)         ento (5.22e-03)   
5        Pont (6.05e-03) ✅  27 (8.39e-03) ✅          bra (4.23e-03)   
6           ega (6.02e-03)  45 (5.93e-03) ✅          ega (3.83e-03)   
7          itar (5.24e-03)   ega (5.34e-03)       Pont (3.81e-03) ✅   
8    levation (3.98e-03) ✅  reta (5.07e-03)          OPT (3.44e-03)   
9           MSS (3.71e-03)  30 (4.87e-03) ✅        ney (3.10e-03) ✅   
10         reta (3.59e-03)    ps (4.53e-03)         heat (2.74e-03)   
11     sector (3.13e-03) ✅  32 (4.38e-03) ✅      bound (2.70e-03) ✅   
12          HEL (3.02e-03)   bon (4.26e-03)        uchar (2.65e-03)   
13      Level (2.88e-03) ✅    de (3.94e-03)       vale (2.57e-03) ✅   
14         olog (2.79e-03)   php (3.78e-03)   Plymouth (2.53e-03) ✅   
15          OPT (2.42e-03)    PS (3.68e-03)       rown (2.50e-03) ✅   
16      Brady (2.35e-03) ✅  24 (3.46e-03) ✅        itori (2.48e-03)   
17          che (2.33e-03)   bat (3.46e-03)         poke (2.28e-03)   
18         Enum (2.29e-03)  ingt (3.43e-03)        gar (2.20e-03) ✅   
19           yt (2.24e-03)   ' ' (3.35e-03)          urt (2.11e-03)   

                      layer_14                          \
                        pos_-3                  pos_-1   
0           elevation (0.0112)     contador (0.5690) ✅   
1          possession (0.0108)              메 (0.1348)   
2            ccording (0.0104)      Produto (0.0165) ✅   
3           welfare (4.56e-03)       zoek (5.44e-03) ✅   
4           booth (4.21e-03) ✅      lokal (4.99e-03) ✅   
5            door (3.78e-03) ✅         spos (4.17e-03)   
6    conversation (3.72e-03) ✅         inne (2.64e-03)   
7           cheer (3.58e-03) ✅          ありが (2.43e-03)   
8              elev (3.37e-03)        cpf (2.27e-03) ✅   
9             grant (3.25e-03)    kontrol (2.25e-03) ✅